# SSL400 ST-GCN — GitHub code + Drive dataset

| What | Where |
|---|---|
| **Model code** | GitHub (`LakmaSanjula/mod-test`) |
| **Dataset CSVs** | Google Drive |
| **Checkpoints / results** | Google Drive (persist after Colab disconnect) |

**Accuracy setup:** `min_samples_per_class=10` (~121 classes) → target ~**64% val / ~62% test**.

1. **Runtime → Change runtime type → GPU (T4)**
2. Run cells in order
3. Push latest code to GitHub from your PC first

## 1) Mount Google Drive (dataset + saved runs only)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) Paths — edit only if your Drive / GitHub layout differs

In [ ]:
from pathlib import Path

# --- GitHub (model code) ---
REPO_URL = "https://github.com/LakmaSanjula/mod-test.git"
REPO_DIR = Path("/content/mod-test")                 # clone location in Colab
PROJECT_DIR = REPO_DIR / "ssl400-research"           # training package inside repo

# --- Google Drive (dataset + persistent outputs) ---
DATASET_DIR = Path("/content/drive/MyDrive/SSL400/Dataset - MP - CSV")
DRIVE_RUNS_DIR = Path("/content/drive/MyDrive/SSL400/runs")  # checkpoints + results

# String paths for later ! shell cells (IPython expands {VAR})
CKPT_DIR = str(DRIVE_RUNS_DIR / "checkpoints")
RESULTS_DIR = str(DRIVE_RUNS_DIR / "results")
PLOTS_DIR = str(DRIVE_RUNS_DIR / "results" / "plots")
PURE_CKPT = str(DRIVE_RUNS_DIR / "checkpoints" / "stgcn_pure_seed42_best.pt")
AUG_CKPT = str(DRIVE_RUNS_DIR / "checkpoints" / "stgcn_augmented_seed42_best.pt")
AUG_H5 = str(DRIVE_RUNS_DIR / "checkpoints" / "stgcn_augmented_seed42_best.h5")
PURE_HIST = str(DRIVE_RUNS_DIR / "results" / "stgcn_pure_seed42_history.json")
AUG_HIST = str(DRIVE_RUNS_DIR / "results" / "stgcn_augmented_seed42_history.json")
PURE_TEST = str(DRIVE_RUNS_DIR / "results" / "stgcn_pure_seed42_test_metrics.json")
AUG_TEST = str(DRIVE_RUNS_DIR / "results" / "stgcn_augmented_seed42_test_metrics.json")

assert DATASET_DIR.exists(), f"Missing dataset on Drive: {DATASET_DIR}"
print("Dataset OK:", DATASET_DIR)
print("Repo URL:", REPO_URL)
print("Project will be:", PROJECT_DIR)
print("Runs will save to:", DRIVE_RUNS_DIR)

## 3) Clone / update model code from GitHub

Public repo: clone works as-is.  
Private repo: set `USE_TOKEN = True` and paste a GitHub PAT when asked.

In [ ]:
import os
from getpass import getpass
from pathlib import Path

USE_TOKEN = False  # True only for a private GitHub repo

clone_url = REPO_URL
if USE_TOKEN:
    token = getpass("GitHub PAT (not your password): ").strip()
    clone_url = REPO_URL.replace("https://", f"https://{token}@")

if (REPO_DIR / ".git").exists():
    print("Repo exists — pulling latest...")
    os.chdir(REPO_DIR)
    !git pull
else:
    print("Cloning repo...")
    !git clone "{clone_url}" "{REPO_DIR}"

assert PROJECT_DIR.exists(), (
    f"Missing {PROJECT_DIR}. Check that ssl400-research is in the GitHub repo root."
)
os.chdir(PROJECT_DIR)
print("CWD:", Path.cwd())
print("Contents:", sorted(p.name for p in PROJECT_DIR.iterdir())[:20])

## 4) Check GPU

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU — switch to T4")
assert torch.cuda.is_available(), "Enable GPU T4 before training"

## 5) Install dependencies

In [ ]:
!pip install -q -r requirements.txt
print("Install done")

## 6) Write configs with Drive/GitHub paths + clean old runs

- `raw_csv_root` → Drive dataset
- `processed_dir` → local Colab disk (faster)
- `checkpoint_dir` / `results_dir` → Drive `SSL400/runs/` (survive disconnect)

In [ ]:
from pathlib import Path

Path("configs").mkdir(exist_ok=True)
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)
ckpt_dir = (DRIVE_RUNS_DIR / "checkpoints").as_posix()
results_dir = (DRIVE_RUNS_DIR / "results").as_posix()
plots_dir = (DRIVE_RUNS_DIR / "results" / "plots").as_posix()

Path("configs/data.yaml").write_text(f'''
raw_csv_root: "{DATASET_DIR.as_posix()}"
processed_dir: "data/processed"
inspected_dir: "data/inspected"
min_samples_per_class: 10
sequence_length: 64
use_z: true
visibility_threshold: 0.5
mask_low_visibility: true
mask_lower_body: false
train_ratio: 0.70
val_ratio: 0.15
test_ratio: 0.15
split_seed: 42
save_tensors: true
'''.strip() + "\n", encoding="utf-8")

pure_yaml = f'''
model:
  in_channels: 3
  num_joints: 33
  num_classes: null
  channels: [64, 64, 128, 256]
  temporal_kernel: 9
  dropout: 0.5
  graph_strategy: spatial

train:
  epochs: 100
  batch_size: 32
  learning_rate: 0.001
  weight_decay: 0.0001
  optimizer: adam
  scheduler: cosine
  early_stopping_patience: 20
  num_workers: 0
  amp: true
  class_weights: false
  weighted_sampler: false
  label_smoothing: 0.0
  focal_gamma: 0.0
  augment: false
  rare_class_augment: false
  seed: 42
  device: auto

paths:
  checkpoint_dir: "{ckpt_dir}"
  results_dir: "{results_dir}"
  plots_dir: "{plots_dir}"
  experiment_name: "stgcn_pure_seed42"
'''.strip() + "\n"

aug_yaml = f'''
model:
  in_channels: 3
  num_joints: 33
  num_classes: null
  channels: [64, 64, 128, 256]
  temporal_kernel: 9
  dropout: 0.5
  graph_strategy: spatial

train:
  epochs: 100
  batch_size: 32
  learning_rate: 0.001
  weight_decay: 0.0001
  optimizer: adam
  scheduler: cosine
  early_stopping_patience: 20
  num_workers: 0
  amp: true
  class_weights: false
  weighted_sampler: false
  label_smoothing: 0.0
  focal_gamma: 1.0
  augment: true
  rare_class_augment: false
  seed: 42
  device: auto

paths:
  checkpoint_dir: "{ckpt_dir}"
  results_dir: "{results_dir}"
  plots_dir: "{plots_dir}"
  experiment_name: "stgcn_augmented_seed42"
'''.strip() + "\n"

Path("configs/model_pure.yaml").write_text(pure_yaml, encoding="utf-8")
Path("configs/model_augmented.yaml").write_text(aug_yaml, encoding="utf-8")
Path("configs/model.yaml").write_text(aug_yaml, encoding="utf-8")

# Clean previous Drive runs + local processed tensors for a correct rebuild
import shutil
for p in (DRIVE_RUNS_DIR / "results", DRIVE_RUNS_DIR / "checkpoints", Path("data/processed")):
    if p.exists():
        shutil.rmtree(p)
        print("Removed", p)
(DRIVE_RUNS_DIR / "results" / "plots").mkdir(parents=True, exist_ok=True)
(DRIVE_RUNS_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)

print("data.yaml raw_csv_root ->", DATASET_DIR)
print("checkpoints ->", ckpt_dir)
print("results ->", results_dir)

## 7) Prepare dataset from Drive CSVs (~121 classes)

In [ ]:
!python scripts/prepare_dataset.py --data-config configs/data.yaml

In [ ]:
import json
from pathlib import Path
import numpy as np

class_map = json.loads(Path("data/processed/class_to_idx.json").read_text(encoding="utf-8"))
manifest = json.loads(Path("data/processed/manifest.json").read_text(encoding="utf-8"))
inv = json.loads(Path("data/inspected/dataset_inventory.json").read_text(encoding="utf-8"))
sample = manifest["train"][0]
tensor = np.load(sample["tensor_path"])

n_classes = len(class_map)
print("Classes:", n_classes)
print("min_samples_per_class:", inv.get("min_samples_per_class"))
print("Split sizes:", {k: len(v) for k, v in manifest.items()})
print("Example:", sample["sample_id"], tensor.shape)

assert inv.get("min_samples_per_class") == 10
assert 100 <= n_classes <= 150, (
    f"Expected ~121 classes for 64% accuracy setup, got {n_classes}"
)
print("OK: accuracy class filter")

## 8A) Train Pure ST-GCN

In [ ]:
!python scripts/train_stgcn.py --data-config configs/data.yaml --model-config configs/model_pure.yaml

In [ ]:
!python scripts/evaluate_stgcn.py \
  --checkpoint "{PURE_CKPT}" \
  --split test \
  --data-config configs/data.yaml \
  --model-config configs/model_pure.yaml

## 8B) Train Augmented ST-GCN (target ~62.5% test)

In [ ]:
!python scripts/train_stgcn.py --data-config configs/data.yaml --model-config configs/model_augmented.yaml

In [ ]:
!python scripts/evaluate_stgcn.py \
  --checkpoint "{AUG_CKPT}" \
  --split test \
  --data-config configs/data.yaml \
  --model-config configs/model_augmented.yaml

## 9) Comparison graphs (saved on Drive)

In [ ]:
!python scripts/compare_runs.py \
  --pure-history "{PURE_HIST}" \
  --aug-history "{AUG_HIST}" \
  --pure-test-metrics "{PURE_TEST}" \
  --aug-test-metrics "{AUG_TEST}" \
  --out-dir "{PLOTS_DIR}"

## 10) MC Dropout + export (augmented)

In [ ]:
!python scripts/mc_dropout_eval.py \
  --checkpoint "{AUG_CKPT}" \
  --split val \
  --passes 20 \
  --data-config configs/data.yaml \
  --model-config configs/model_augmented.yaml

!python scripts/export_h5.py \
  --checkpoint "{AUG_CKPT}" \
  --output "{AUG_H5}"

## 11) Show graphs from Drive

In [ ]:
from pathlib import Path
from IPython.display import Image, display

plots_dir = DRIVE_RUNS_DIR / "results" / "plots"
wanted = [
    "stgcn_pure_seed42_loss_curves.png",
    "stgcn_augmented_seed42_loss_curves.png",
    "compare_pure_vs_augmented_loss.png",
    "compare_pure_vs_augmented_val_metrics.png",
    "compare_pure_vs_augmented_best_metrics.png",
]
for name in wanted:
    path = plots_dir / name
    if path.exists():
        print("Showing", path)
        display(Image(filename=str(path)))
    else:
        print("Missing:", path)

## 12) Summary

In [ ]:
import json
from pathlib import Path

results = DRIVE_RUNS_DIR / "results"
for exp in ("stgcn_pure_seed42", "stgcn_augmented_seed42"):
    s_path = results / f"{exp}_train_summary.json"
    m_path = results / f"{exp}_test_metrics.json"
    print("=" * 60)
    print(exp)
    if s_path.exists():
        s = json.loads(s_path.read_text())
        print("classes", s.get("num_classes"), "val_acc", s.get("best_val_accuracy"), "val_f1", s.get("best_val_f1_macro"))
    if m_path.exists():
        m = json.loads(m_path.read_text())
        print("test_acc", m.get("accuracy"), "test_f1", m.get("f1_macro"), "top5", m.get("top5_accuracy"))

print("\nCode came from GitHub:", REPO_URL)
print("Dataset on Drive:", DATASET_DIR)
print("Outputs on Drive:", DRIVE_RUNS_DIR)